In [1]:
import numpy as np
from scipy import stats

# ==============================================================================
#(Варіант 6)
# ==============================================================================
CITY = "Chernihiv"
REGION = "Center"
NORMA = 75

# Ранкова вологість о 8:00 (12 місяців)
morning = np.array([85, 83, 79, 71, 66, 63, 61, 63, 69, 76, 82, 86])

# Добова амплітуда для віднімання (вечірні вимірювання)
amplitude = np.array([3, 3, 4, 5, 6, 7, 7, 7, 6, 5, 4, 3])


# ==============================================================================
# ЗАВДАННЯ 1. Одновибірковий t-критерій проти норми
# ==============================================================================
print("--- Task 1 ---")

# H0: mu = 75 (Середня вологість у Чернігові відповідає багаторічній нормі)
# H1: mu != 75 (Середня вологість у Чернігові статистично значуще відрізняється від норми)

t_stat1, p_val1 = stats.ttest_1samp(morning, popmean=NORMA)

print(f"t-statistic: {t_stat1:.4f}")
print(f"p-value: {p_val1:.4f}")

# ВИСНОВОК ДО ЗАВДАННЯ 1:
# Оскільки p-value = 0.6171 > 0.05 (alpha = 0.05), ми не маємо підстав відхилити H0.
# Немає статистично значущих доказів того, що середня вологість повітря
# у Чернігові за цей рік відрізняється від норми у 75%.


# ==============================================================================
# ЗАВДАННЯ 2. Ручна перевірка t-статистики
# ==============================================================================
print("\n--- Task 2 ---")

mean_x = morning.mean()
s_x = morning.std(ddof=1)
n = len(morning)

t_manual = (mean_x - NORMA) / (s_x / np.sqrt(n))

print(f"Mean: {mean_x:.4f}")
print(f"Std (ddof=1): {s_x:.4f}")
print(f"Manual t-statistic: {t_manual:.4f}")
print(f"Match: {np.isclose(t_stat1, t_manual)}")

# ВИСНОВОК ДО ЗАВДАННЯ 2:
# Значення t-статистики, обчислене вручну (-0.5139), з точністю до округлення
# повністю збігається із значенням t_stat1 (-0.5139) з функції ttest_1samp.


# ==============================================================================
# ЗАВДАННЯ 3. Незалежний двовибірковий критерій: Чернігів проти Харків (Схід)
# ==============================================================================
print("\n--- Task 3 ---")

# Варіант 4: Харків (Схід)
kharkiv = np.array([83, 80, 75, 68, 62, 60, 58, 60, 66, 73, 79, 84])

# Крок 1: Критерій Лівена для перевірки рівності дисперсій
# H0: Дисперсії рівні (гетероскедастичність відсутня)
# H1: Дисперсії статистично відрізняються
stat_lev3, p_lev3 = stats.levene(morning, kharkiv)
print(f"Levene test p-value: {p_lev3:.4f}")

# Оскільки p_lev3 = 0.9589 >= 0.05, вважаємо дисперсії двох міст рівними (гомогенними).
equal_var_flag3 = p_lev3 >= 0.05

# Крок 2: Оскільки дисперсії рівні, використовуємо стандартний t-критерій (equal_var=True)
t_stat3, p_val3 = stats.ttest_ind(morning, kharkiv, equal_var=equal_var_flag3)
print(f"t-statistic: {t_stat3:.4f}")
print(f"p-value: {p_val3:.4f}")

# ВИСНОВОК ДО ЗАВДАННЯ 3:
# За тестом Лівена (p = 0.9589 >= 0.05) підстав вважати дисперсії нерівними немає,
# тому обґрунтовано застосовано equal_var=True.
# За результатом ttest_ind (p = 0.4191 > 0.05) між середньою вологістю
# Чернігова та Харкова немає статистично значущої різниці.


# ==============================================================================
# ЗАВДАННЯ 4. Порівняння з протилежним висновком Лівена: Чернігів проти Одеса (Південь)
# ==============================================================================
print("\n--- Task 4 ---")

# Варіант 3: Одеса (Південь)
odesa = np.array([82, 80, 78, 74, 71, 69, 67, 68, 72, 76, 80, 83])

# Крок 1: Перевірка рівності дисперсій
stat_lev4, p_lev4 = stats.levene(morning, odesa)
print(f"Levene test p-value: {p_lev4:.4f}")

# Оскільки p_lev4 = 0.0411 < 0.05, нульову гіпотезу відхиляємо:
# дисперсії вибірок Чернігова та Одеси є статистично НЕРІВНИМИ.
equal_var_flag4 = p_lev4 >= 0.05

# Крок 2: Застосовуємо Welch-варіант (equal_var=False) через нерівність дисперсій
t_stat4, p_val4 = stats.ttest_ind(morning, odesa, equal_var=equal_var_flag4)
print(f"t-statistic: {t_stat4:.4f}")
print(f"p-value: {p_val4:.4f}")

# ПОЯСНЕННЯ ТА ОБҐРУНТУВАННЯ ДО ЗАВДАННЯ 4:
# У Завданні 3 тест Лівена дав p >= 0.05, що дозволило використати класичний
# t-критерій Стьюдента з об'єднаною оцінкою дисперсії (equal_var=True).
# У Завданні 4 тест Лівена дав p = 0.0411 < 0.05, що свідчить про суттєву
# нерівність дисперсій (в Одесі розкид 5.38%, у Чернігові 8.99%).
# Тому тут обов'язково застосовано поправку Уелча (equal_var=False),
# яка коригує кількість ступенів свободи для уникнення викривлення p-value.
# За результатом ttest_ind (p = 0.6631 > 0.05) значущої різниці між середніми немає.


# ==============================================================================
# ЗАВДАННЯ 5. Парний t-критерій: ранок проти вечора
# ==============================================================================
print("\n--- Task 5 ---")

evening = morning - amplitude
print("Evening array:", evening)

t_stat5, p_val5 = stats.ttest_rel(morning, evening)
print(f"Paired t-statistic: {t_stat5:.4f}")
print(f"p-value: {p_val5:.4e}")

# ВИСНОВОК ТА ПОЯСНЕННЯ ДО ЗАВДАННЯ 5:
# 1. Висновок: p-value = 1.43e-7 << 0.05, тому H0 відхиляємо.
#    Існує високозначуща статистична різниця між ранковою та вечірньою вологістю
#    (вранці вона систематично вища).
# 2. Пояснення, чому потрібен ПАРНИЙ критерій:
#    Дані виміряні в однакових містах і в ті самі місяці. Парний критерій
#    обчислює різницю (d_i = ранок_i - вечір_i) для кожного місяця окремо.
#    Це дозволяє повнісю УСУНУТИ міжмісячну (сезонну) варіабельність
#    (велику різницю між зимою та літом), порівнюючи лише добове коливання.


#ВІДПОВІДІ НА КОНТРОЛЬНІ ПИТАННЯ:

#1. Чому використовується t-, а не z-розподіл?
#   z-розподіл вимагає знання генеральної дисперсії (sigma). У реальних дослідженнях
#   она невідома і оцінюється за вибірковим стандартним відхиленням (s).
#   Оцінювання s вносить додаткову невизначеність. t-розподіл Стьюдента враховує
#  цю додаткову помилку вибірки і має "важчі хвости" при малих обсягах (n=12),
#   що запобігає хибнопозитивним висновкам.

#2. Чому неправильний вибір equal_var дає систематично неправильний p-value?
#   При equal_var=True використовується формула об'єднаної дисперсії. Якщо дисперсії
#   вибірок сильно відрізняються, ця формула некоректно обчислює стандартну помилку
#   і завищує кількість ступенів свободи. Це спотворює форму розподілу критерію
#   і призводить до викривлення p-value (підвищується ризик похибки I або II типу).
#   Поправка Уелча (equal_var=False) коригує ступені свободи та стандартну помилку.

#3. Чому парний критерій потужніший за незалежний?
#   Парний критерій працює з масивом індивідуальних різниць (d_i = X_i - Y_i).
#   При цьому повністю видаляється фоновий шум — міжсуб'єктна або сезонна варіація.
#   У випадку вологості різниця між зимою та літом дуже велика. Якщо загнорувати
#   паруваність і застосувати незалежний критерій, сезонний розкид збільшить
#   загальну дисперсію і "приглушить" реальний ефект добової зміни.

#4. Узгодженість формального критерію Лівена зі std() "на око":
#   Так, висновки повністю узгоджуються:
#   - Завдання 3 (Чернігів vs Харків): s_Чернігів = 8.99%, s_Харків = 8.87%.
#     Відхилення майже однакові "на око", і тест Лівена це підтвердив (p = 0.9589).
#   - Завдання 4 (Чернігів vs Одеса): s_Чернігів = 8.99%, s_Одеса = 5.38%.
#     В Одесі вологість протягом року значно стабільніша, що чітко видно за std()
#     і формально підтверджено тестом Лівена (p = 0.0411 < 0.05).


--- Task 1 ---
t-statistic: -0.4979
p-value: 0.6284

--- Task 2 ---
Mean: 73.6667
Std (ddof=1): 9.2769
Manual t-statistic: -0.4979
Match: True

--- Task 3 ---
Levene test p-value: 0.9152
t-statistic: 0.7807
p-value: 0.4433

--- Task 4 ---
Levene test p-value: 0.0158
t-statistic: -0.4257
p-value: 0.6753

--- Task 5 ---
Evening array: [82 80 75 66 60 56 54 56 63 71 78 83]
Paired t-statistic: 10.8562
p-value: 3.2341e-07
